In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd
import numpy as np

# 1. Memuat Dataset Mentah
df = pd.read_csv('/content/drive/MyDrive/ExploratoryDataAnalysis/Pertemuan5/transaksi_koperasi_2025.csv')
print(f"Jumlah baris awal: {len(df)}")

# 2. Perbaiki Tipe Data & Format Kolom 'total_bayar'
# Menghilangkan teks 'Rp', titik ribuan, dan mengubah ke tipe numerik
df['total_bayar'] = (
    df['total_bayar']
    .astype(str)
    .str.replace('Rp', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.strip()
)
df['total_bayar'] = pd.to_numeric(df['total_bayar'], errors='coerce')

# 3. Buang Spasi pada Kolom Teks & Seragamkan Kategori
# Membersihkan spasi berlebih dan menyamakan kapitalisasi kategori
df['kategori'] = df['kategori'].astype(str).str.strip().str.title()
# Memperbaiki typo dan menyatukan kategori yang mirip (misal: Alat Tulis -> ATK, Minumam -> Minuman)
df['kategori'] = df['kategori'].replace({
    'Alat Tulis': 'ATK',
    'Minumam': 'Minuman'
})

# 4. Tangani Placeholder & Nilai Metode Pembayaran
df['metode_bayar'] = df['metode_bayar'].astype(str).str.strip().str.title()
# Mengubah placeholder seperti '-', 'Nan', atau 'Tidak Diketahui' menjadi NaN yang jujur
df.loc[df['metode_bayar'].isin(['-', 'Nan', 'Tidak Diketahui']), 'metode_bayar'] = np.nan
df['metode_bayar'] = df['metode_bayar'].replace({'Qris': 'QRIS'})

# 5. Tangani Duplikat
df = df.drop_duplicates()

# 6. Tangani Nilai Mustahil (Anomali Logika)
# - Jumlah barang beli tidak boleh negatif (<= 0)
# - Diskon persen harus berada di rentang 0 hingga 100
# - Rating layanan tidak boleh bernilai ekstrem/placeholder (-999)
df = df[df['jumlah'] > 0]
df = df[(df['diskon_persen'] >= 0) & (df['diskon_persen'] <= 100)]
df = df[df['rating_layanan'] != -999]

# 7. Simpan Hasil Bersih ke Direktori Terpisah
# Pastikan folder 'data/processed/' sudah dibuat sebelumnya
import os
os.makedirs('data/processed', exist_ok=True)
df.to_csv('data/processed/transaksi_koperasi_bersih.csv', index=False)

print(f"Jumlah baris setelah dibersihkan: {len(df)}")
print("Pembersihan selesai dan file tersimpan di data/processed/!")

Jumlah baris awal: 1543
Jumlah baris setelah dibersihkan: 1437
Pembersihan selesai dan file tersimpan di data/processed/!
